# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)


In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

import pandas as pd  # noqa: E402
import numpy as np  # noqa: E402

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Loaded:", df.shape)


Working dir: /root/FlyRank-Internship-ML
Loaded: (30000, 44)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page), identified by `content_id`, belonging to one of 32
pseudonymized clients (`client_id`). Each row is a trailing-90-day snapshot ending at export —
not a time series. That matters: this data can show *that* a page trended down over the last
90 days, but never *when* the drop happened, day by day. Verified below.

In [2]:
print("rows:", len(df), "| distinct content_id:", df["content_id"].nunique())
dup = df.groupby("content_id").size()
print("content_id appearing more than once:", (dup > 1).sum())

print("\ncontent_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())
print("distinct client_id:", df["client_id"].nunique())


rows: 30000 | distinct content_id: 30000
content_id appearing more than once: 0

content_age_days min/max: 90 564
distinct client_id: 32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Label** — `trend_direction` / `trend_pct` (`is_declining_label = trend_direction == "down"`).
  Never a feature.
- **Excluded because they leak into the label**: `impressions_last_30d`, `clicks_last_30d`,
  `sessions_last_30d` — the label is built from a last-30-vs-prev-30 comparison, so these three
  columns are literally half of that comparison's input.
- **Excluded, other**: `provider_used`, `model_used` — which tool generated the article isn't a
  property of the page's search performance; keeping them risks the model keying off
  content-provenance instead of content quality.
- **Context** — `content_id` (unique key, never learned from), `client_id` (never a raw feature:
  using it directly would let the model memorize per-client behavior instead of general
  patterns. Used instead to drive a client-holdout train/test split, so a client's typical
  behavior can't leak between train and test the way it could with a random row split).
- **Feature** — everything else: search volume/competition/cpc, content_type/main_intent,
  word_count/char_count and their tiers, age/freshness fields, the `_90d` and `_prev_30d`
  activity counts, and their derived rates (`ctr`, `avg_position`, `engagement_rate`,
  `scroll_rate`, `ai_traffic_pct`) — all knowable before the label's last-30-day window closes.

In [3]:
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]
excluded_cols = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "provider_used", "model_used",
]
feature_cols = [c for c in df.columns if c not in label_cols + context_cols + excluded_cols]

print("label:", label_cols)
print("context:", context_cols)
print("excluded:", excluded_cols)
print("feature count:", len(feature_cols))
print("total classified:", len(label_cols + context_cols + excluded_cols + feature_cols),
      "/ total columns:", df.shape[1])


label: ['trend_direction', 'trend_pct']
context: ['content_id', 'client_id']
excluded: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'provider_used', 'model_used']
feature count: 35
total classified: 44 / total columns: 44


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain and window claims were already verified in section 1. What's left: missingness. Checked
below — it isn't random, it follows `content_type`, and the two fields follow *different*
content types (opposite of each other), which is why a blind `fillna(0)` would be wrong for
either one.

In [4]:
miss = df.isna().mean().sort_values(ascending=False)
print("overall missingness (nonzero columns):")
print(miss[miss > 0].round(3))

print("\nsearch_volume missing by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3))

print("\nword_count missing by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3))


overall missingness (nonzero columns):
provider_used        0.715
char_count           0.257
word_count           0.257
word_count_tier      0.257
char_count_tier      0.257
model_used           0.191
trend_pct            0.113
competition_level    0.087
cpc                  0.082
competition          0.082
search_volume        0.082
main_intent          0.079
scroll_rate          0.004
dtype: float64

search_volume missing by content_type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

word_count missing by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No true time series.** One snapshot per page — shows *that* a page trended down, never
  *when*, day by day.
- **32 clients, but wildly uneven.** Rows per client range from 3 to 7,008 (verified below) —
  any aggregate computed without grouping by `client_id` first is really describing the handful
  of clients that dominate the row count, not "the dataset."
- **No brand-new content.** `content_age_days` never drops below 90 and `age_tier` never shows
  `0-14` or `15-30` — this slice says nothing about how content behaves in its first three
  months.

In [5]:
print("rows per client — min/max spread:")
print(df.groupby("client_id").size().sort_values().iloc[[0, -1]])

print("\nage_tier values present (should exclude 0-14, 15-30):")
print(sorted(df["age_tier"].unique()))


rows per client — min/max spread:
client_id
client_1a6562590e       3
client_19581e27de    7008
dtype: int64

age_tier values present (should exclude 0-14, 15-30):
['181-365', '31-90', '365+', '91-180']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.